In [1]:
import mdtraj as md
import numpy as np

from tqdm import tqdm
from glob import glob

from matplotlib import pyplot as plt
from matplotlib.ticker import FuncFormatter

from pathlib import Path

from sklearn.cross_decomposition import CCA

In [2]:
def sci_notation_no_sigfigs(number, pos):
    a, b = f"{number:.0e}".split("e")
    b = int(b)
    if b == 0:
        return "0"
    return f"${a} \, \mathrm{{x}} \, 10^{{{b}}}$"


def sci_notation_one_sigfig(number, pos):
    a, b = f"{number:.1e}".split("e")
    b = int(b)
    if b == 0:
        return "0"
    return f"${a} \, \mathrm{{x}} \, 10^{{{b}}}$"

## Load MD and Crystal data

In [3]:
def load_pdbs(glob_pattern, atom_selection='(element != H) and is_protein', Ca_only=True):
    
    crystals = None
    files_to_load = sorted(glob(glob_pattern))
    
    for pdb in tqdm(files_to_load):

        try:
            
            trj = md.load_pdb(pdb)
            idx = trj.top.select(ATOM_SELECTION)
            trj = trj.atom_slice(idx)

            if Ca_only:
                idx = trj.top.select('name == CA')
                trj = trj.atom_slice(idx)

            if crystals:
                crystals = crystals.join(trj, check_topology=False)
            else:
                crystals = trj

        except Exception as exptn:
            print(trj.xyz.shape)
            print(crystals.xyz.shape)
            print("something went wrong", pdb, exptn)

    #crystals = crystals.center_coordinates()
    crystals = crystals.superpose(crystals[0])
    
    return crystals


def load_simulation(glob_pattern, top_path, subsampling=1, atom_selection='(element != H) and is_protein', Ca_only=True):

    md_files = sorted(glob(glob_pattern))
    print(f"found {len(md_files)} trajectory files...")
    
    md_sim = None

    for f in tqdm(md_files):
    
        t = md.load(f, top=top_path)[::subsampling]
    
        idx = t.top.select(atom_selection)
        t   = t.atom_slice(idx)

        if Ca_only:
            idx = t.top.select('name == CA')
            t   = t.atom_slice(idx)
    
        if md_sim == None:
            md_sim = t
        else:
            md_sim += t
    
    md_sim = md_sim.center_coordinates()
    
    return md_sim

In [4]:
ATOM_SELECTION = '(element != H) and is_protein and (resi < 303) and (resi > 0)'

In [5]:
crystals = load_pdbs('../../allostery/selected_dataset_archive_2024-08-07/pdb/*.pdb', atom_selection=ATOM_SELECTION)

100%|██████████| 1146/1146 [04:14<00:00,  4.50it/s]


In [6]:
prefix = '/gpfs/cfel/user/tjlane/mpro/allostery/diamond_05-May-2020/'
diamond = load_pdbs(prefix + '*.pdb', atom_selection=ATOM_SELECTION)

 58%|█████▊    | 61/105 [00:11<00:07,  5.64it/s]

(1, 300, 3)
(59, 302, 3)
something went wrong /gpfs/cfel/user/tjlane/mpro/allostery/diamond_05-May-2020/Mpro-x1187_0.pdb Number of atoms in self (302) is not equal to number of atoms in other


 83%|████████▎ | 87/105 [00:15<00:02,  6.55it/s]

(1, 302, 3)
(84, 302, 3)
something went wrong /gpfs/cfel/user/tjlane/mpro/allostery/diamond_05-May-2020/Mpro-x2052_0.pdb index 0 is out of bounds for axis 0 with size 0


100%|██████████| 105/105 [00:18<00:00,  5.57it/s]


In [7]:
# phenix ensembles

phenix_ensembles = []
ensemble_pdb_paths = glob('/gpfs/cfel/user/tjlane/mpro/allostery/ensemble_refinements/four_datasets/*/*.updated_ensemble.nowater.pdb')

for ensemble_pdb_path in ensemble_pdb_paths:
    ensemble = load_pdbs(ensemble_pdb_path, atom_selection=ATOM_SELECTION)
    phenix_ensembles.append(ensemble)

100%|██████████| 1/1 [00:05<00:00,  5.49s/it]


In [10]:
# --- simulations ---

# DESRES
# sampling is 1 ns, so subsampled 1/1 is 1 ns

prefix = '/asap3/petra3/gpfs/p11/2020/data/11009999/shared/mdsim/DESRES-Trajectory_sarscov2-10880334-no-water-no-ion-glueCA'
glob_pattern = prefix + '/sarscov2-10880334-no-water-no-ion-glueCA/*.dcd'
top_path = prefix + '/system_nowat.pdb'

desres_sim = load_simulation(glob_pattern, top_path, subsampling=1, atom_selection=ATOM_SELECTION)

# RIKEN
# sampling is 200 ps, so subsampled 1/5 is 1 ns

prefix = '/gpfs/cfel/user/tjlane/mpro/mpro-simulations/RIKEN_mpro_simulation'
glob_pattern = prefix + '/Traj?/protein_snap_every200ps_*.xtc'
top_path = prefix + '/Traj1/protein_conf.gro'

riken_sim = load_simulation(glob_pattern, top_path, subsampling=5, atom_selection=ATOM_SELECTION)

# GaMD
# 200 ps sampling

prefix = '/gpfs/cfel/user/tjlane/mpro/mpro-simulations/TRAJECTORIES_main_protease_6LU7_amarolab/6LU7_dimer_apo'

glob_pattern_combined = prefix + '/*/6LU7_dimer_apo.nc'
top_path_combined = prefix + '/6LU7_dimer_apo.prmtop'

amaro_sim = load_simulation(glob_pattern_combined, top_path_combined, subsampling=5, atom_selection=ATOM_SELECTION)
amaro_sim = amaro_sim.superpose(amaro_sim[0])

  0%|          | 0/100 [00:00<?, ?it/s]

found 100 trajectory files...


  0%|          | 0/6 [00:00<?, ?it/s]

found 6 trajectory files...


  0%|          | 0/5 [00:00<?, ?it/s]

found 5 trajectory files...


100%|██████████| 5/5 [00:04<00:00,  1.04it/s]


In [11]:
print(crystals.xyz.shape)
print(diamond.xyz.shape)
print([ensemble.xyz.shape for ensemble in phenix_ensembles])

print(desres_sim.xyz.shape)
print(riken_sim.xyz.shape)
print(amaro_sim.xyz.shape)

(1146, 302, 3)
(103, 302, 3)
[(67, 302, 3), (29, 302, 3), (50, 302, 3), (50, 302, 3)]
(100000, 302, 3)
(20000, 302, 3)
(1000, 302, 3)


In [12]:
active_site_residues = np.array([41, 49, 143, 144, 145, 163, 164, 165, 166, 167, 187, 188, 189, 190, 191, 192]) - 2

residues = list(crystals.top.residues)
asr_names = [ residues[asr].name + str(residues[asr].resSeq) for asr in active_site_residues ]

print(asr_names)

['HIS41', 'MET49', 'GLY143', 'SER144', 'CYS145', 'HIS163', 'HIS164', 'MET165', 'GLU166', 'LEU167', 'ASP187', 'ARG188', 'GLN189', 'THR190', 'ALA191', 'GLN192']


## canonical correlation

In [13]:
def canonical_correlation_analysis(xyz):
    """
    CCA finds the vectors a and b that maximze corr(ax, by)
    
    here, x and y are samples each atom's xyz position 3-vectors
    
    this function returns the coefficients of determination R^2 for each atom pair i,j
    """
    
    cca = np.zeros((xyz.shape[1], xyz.shape[1]))
    
    for atom_index_i in range(xyz.shape[1]):
        for atom_index_j in range(atom_index_i, xyz.shape[1]):
            
            x = xyz[:,atom_index_i,:]
            y = xyz[:,atom_index_j,:]
            
            r2 = CCA(n_components=1).fit(x, y).score(x, y)
            
            cca[atom_index_i, atom_index_j] = r2
            cca[atom_index_j, atom_index_i] = r2
    
    return cca

In [ ]:
S_xtal = canonical_correlation_analysis(crystals.xyz)
S_dmnd = canonical_correlation_analysis(diamond.xyz)
S_ensb = [canonical_correlation_analysis(ensemble.xyz) for ensemble in phenix_ensembles]

S_desres = canonical_correlation_analysis(desres_sim.xyz)
S_riken  = canonical_correlation_analysis(riken_sim.xyz)

S_amaro  = canonical_correlation_analysis(amaro_sim.xyz)

In [ ]:
#err_S_xtal = bootstrap_iso_cov(crystals.xyz, n_samples=1000)

In [ ]:
# title, matrix to plot, filename to save
phenix_ensemble_datasets = [
    ('phenix ensemble %d' % (ip+1), S_ensb[ip], f"ensemble_{Path(dataset_name).name.split('.')[0]}") for ip, dataset_name in enumerate(ensemble_pdb_paths)
]
phenix_ensemble_datasets

In [ ]:
ticks = np.arange(0, crystals.xyz.shape[1], 100)
cmap = 'viridis'

# title, matrix to plot, filename to save
datasets = [
    (r'ligand-free M$^\mathrm{pro}$ crystals', S_xtal, "cov_xtal_ensemble"),
    ('Diamond Frag. Screen', S_dmnd, "cov_dmnd_ensemble"),
    ('DESRES Simulation', S_desres[:,:], "desres_sim"),
    ('RIKEN Simulation', S_riken[:,:], "riken_sim"),
    ('Amaro Simulation', S_amaro[:,:], "amaro_sim"),
]


# how much to zoom colorbar
scales = [0.8, 0.6, 0.4, 1.1, 0.3]

# datasets = [(r'ligand-free M$^\mathrm{pro}$ crystals', S_xtal, "cca_xtal_ensemble"),]
# scales = [0.7,]

for i_d, dataset in enumerate(datasets):

    figsize = (3.5,3.5)

    fig = plt.figure(figsize=figsize)
    ax1 = plt.subplot(111)

    ax1.set_title(dataset[0], 
                  fontweight='bold', fontsize=10)
    

    print(dataset[1].max())
    im = ax1.imshow(
        dataset[1],
        cmap=cmap, 
        vmin=0,
        vmax=scales[i_d],
    )

    ax1.set_xticks(ticks)
    ax1.set_xticklabels([str(int(t)) for t in ticks])
    ax1.set_yticks(ticks)
    ax1.set_yticklabels([str(int(t)) for t in ticks])
    ax1.set_xlabel('Residue Index')
    ax1.set_ylabel('Residue Index')
    
    # ax1.set_xticks([])
    # ax1.set_yticks([])

    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    
    cbar.set_label('CCA R$^2$ ($\AA^2$)', rotation=270, labelpad=15)

    plt.tight_layout()
    plt.savefig(f'./figures/covariance_matrices/cca_{dataset[2]}.pdf')
    plt.show()

## View correlation projected on active site

In [ ]:
mu_xtal  = np.abs(S_xtal[active_site_residues,:]).mean(0)
std_xtal = np.abs(S_xtal[active_site_residues,:]).std(0)

In [ ]:
res_idx = np.arange(S_xtal.shape[0]) + 3

plt.figure(figsize=(5,3))

ax1 = plt.subplot(111)

ax1.plot(res_idx, mu_xtal, color='b')

ax1.set_xlabel(r'Residue Index')
ax1.set_ylabel(r'Average Active Site CCA R$^2$')

# interesting
ymin = 0.0
ymax = 0.75
y_shift = 0.1

bbox = dict(facecolor='white', edgecolor='black')
ax1.text(194, ymax - y_shift, 'N214', weight='bold', bbox=bbox)
ax1.text(236, ymax - 2*y_shift, 'Q256', weight='bold', bbox=bbox)
ax1.text(264, ymax - 3*y_shift, 'S284', weight='bold', bbox=bbox)

ax1.vlines(214, ymin, ymax, linestyles="--")
ax1.vlines(256, ymin, ymax, linestyles="--")
ax1.vlines(284, ymin, ymax, linestyles="--")

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

    
plt.tight_layout()
plt.savefig('figures/cca_to_active.pdf')
plt.show()